<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/05-files-and-data/04_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: From Raw CSV to a Saved Penguin Classifier

In this project you'll take the raw Palmer Penguins file and go all the way to a trained classifier that's saved to disk, loaded back, and used to make predictions for new penguins. That's the full life cycle of a real model, and every step involves files.

You'll build:

1. a loader that reads and validates the raw CSV
2. a reproducible, stratified train/test split
3. a **Gaussian Naive Bayes** classifier, a new algorithm that's a natural follow-on from the spam filter in Module 2
4. an evaluation with a confusion matrix, compared against two other models
5. a model file in plain JSON, including a fingerprint of the exact data it was trained on
6. a prediction step that reads new observations from a CSV, handles missing measurements, and writes predictions with probabilities to a new CSV

Pure Python, standard library only.

## Step 1: Get and load the data

Same download-once function as in the lesson, and a loader that converts types and reports anything unexpected:

In [1]:
import csv
import hashlib
import json
import math
import random
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/anupbagale/python-for-ai-ml/main/05-files-and-data/data/penguins.csv"
DATA_DIR = Path("data")
WORK = Path("work")


def get_penguins() -> Path:
    path = DATA_DIR / "penguins.csv"
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"downloading {DATA_URL}")
        urllib.request.urlretrieve(DATA_URL, path)
    return path


FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
MISSING = {"", "na", "n/a", "nan", "null", "none", "?", "-"}


def to_number(text: str) -> float | None:
    text = text.strip()
    return None if text.lower() in MISSING else float(text)


def load_rows(path: Path) -> list[dict]:
    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        needed = {"species", "island"} | set(FEATURES)
        absent = needed - set(reader.fieldnames)
        if absent:
            raise ValueError(f"{path} is missing columns: {sorted(absent)}")
        rows = []
        for raw in reader:
            row = {"species": raw["species"].strip(), "island": raw["island"].strip()}
            for col in FEATURES:
                row[col] = to_number(raw[col])
            rows.append(row)
    return rows


raw_path = get_penguins()
rows = load_rows(raw_path)
print(len(rows), "rows")
print(Counter(r["species"] for r in rows))

344 rows
Counter({'Adelie': 152, 'Gentoo': 124, 'Chinstrap': 68})


The loader checks the header before reading any rows. If someone hands you a file with a renamed column, you get a clear error naming the missing column right away, rather than a `KeyError` somewhere in the middle of training.

It opens the file with `"utf-8-sig"`, so it also works if the CSV has been opened and re-saved in Excel.

## Step 2: A fingerprint of the data

Before using the data, we compute a **checksum**: a short string calculated from every byte of the file. If even one character in the file changes, the checksum changes completely. We'll store it in the saved model, so that later, anyone can check exactly which data the model was trained on.

In [2]:
def sha256_of(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()


data_hash = sha256_of(raw_path)
print(data_hash)

f204db2c753b0937caac3cb35258562c14f073e4bbc76be24b4c51ce22767a93


Two details:

- The file is opened in binary mode (`"rb"`), because a checksum is about the exact bytes, not the text they represent.
- It's read in chunks of 64 KB, so this works on multi-gigabyte files without loading them into memory. `iter(function, sentinel)` is a lesser-known form of `iter` that keeps calling the function until it returns the sentinel value, here the empty bytes `b""` that `read` returns at the end of the file.

Data versioning tools like DVC are built around exactly this idea: identify every dataset by a hash of its contents.

## Step 3: Clean and split

As we found in the lesson, two penguins have no measurements at all, so we drop them. Then a stratified split with a fixed seed, so every species is represented in the same proportion in both sets, and the split is identical every time the notebook runs.

In [3]:
usable = [r for r in rows if all(r[c] is not None for c in FEATURES)]
print(f"dropped {len(rows) - len(usable)} rows with missing measurements")


def stratified_split(rows, key, test_ratio, seed):
    groups = defaultdict(list)
    for row in rows:
        groups[row[key]].append(row)
    rng = random.Random(seed)
    train, test = [], []
    for label in sorted(groups):
        group = groups[label][:]
        rng.shuffle(group)
        n_test = round(len(group) * test_ratio)
        test += group[:n_test]
        train += group[n_test:]
    return train, test


SEED = 7
train, test = stratified_split(usable, "species", test_ratio=0.25, seed=SEED)
print("train:", len(train), dict(Counter(r["species"] for r in train)))
print("test: ", len(test), dict(Counter(r["species"] for r in test)))

X_train = [[r[c] for c in FEATURES] for r in train]
y_train = [r["species"] for r in train]
X_test = [[r[c] for c in FEATURES] for r in test]
y_test = [r["species"] for r in test]

dropped 2 rows with missing measurements
train: 256 {'Adelie': 113, 'Chinstrap': 51, 'Gentoo': 92}
test:  86 {'Adelie': 38, 'Chinstrap': 17, 'Gentoo': 31}


## Step 4: Gaussian Naive Bayes

The spam filter in Module 2 was a Naive Bayes classifier for **words**: it counted how often each word appeared in each class. Our features here are **measurements**, which are continuous numbers, so we can't count them. Instead, Gaussian Naive Bayes assumes that within each species, each measurement follows a bell curve (a *Gaussian*, or *normal*, distribution).

Training learns three things per species:

- the **prior**: what fraction of penguins are this species
- the **mean** of each measurement for this species
- the **variance** of each measurement for this species, which describes how spread out it is

To classify a new penguin, we ask, for each species: how likely is it to see these exact measurements if the penguin belonged to that species? A bell curve gives that likelihood for one measurement:

$$p(x \mid \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

Measurements close to the species' mean get a high likelihood, and ones far away get a very low one. Like the spam filter, the model is "naive": it treats the measurements as independent and multiplies their likelihoods together. And like the spam filter, we add **logarithms** instead of multiplying, to avoid underflow. The log of the formula above is:

$$\log p = -\frac{1}{2}\log(2\pi\sigma^2) - \frac{(x - \mu)^2}{2\sigma^2}$$

The species with the highest total score (log prior plus the log-likelihoods of all its measurements) wins.

The model also needs to be saved and loaded, so we give it `to_dict` and `from_dict` methods that convert it to and from plain JSON-friendly data, the class method pattern from Module 3.

In [4]:
class GaussianNB:
    def __init__(self, var_smoothing: float = 1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X: list[list[float]], y: list[str]) -> "GaussianNB":
        self.classes_ = sorted(set(y))
        n_features = len(X[0])
        largest_var = max(self._variance([row[j] for row in X]) for j in range(n_features))
        epsilon = self.var_smoothing * largest_var

        self.priors_, self.means_, self.vars_ = {}, {}, {}
        for c in self.classes_:
            rows = [x for x, label in zip(X, y) if label == c]
            self.priors_[c] = len(rows) / len(X)
            columns = list(zip(*rows))
            self.means_[c] = [sum(col) / len(col) for col in columns]
            self.vars_[c] = [self._variance(col) + epsilon for col in columns]
        return self

    @staticmethod
    def _variance(values) -> float:
        m = sum(values) / len(values)
        return sum((v - m) ** 2 for v in values) / len(values)

    def log_scores(self, x: list[float | None]) -> dict[str, float]:
        scores = {}
        for c in self.classes_:
            score = math.log(self.priors_[c])
            for value, mean, var in zip(x, self.means_[c], self.vars_[c]):
                if value is None:
                    continue                      # a missing measurement just adds no evidence
                score += -0.5 * math.log(2 * math.pi * var) - (value - mean) ** 2 / (2 * var)
            scores[c] = score
        return scores

    def predict_proba(self, x: list[float | None]) -> dict[str, float]:
        scores = self.log_scores(x)
        top = max(scores.values())
        exps = {c: math.exp(s - top) for c, s in scores.items()}
        total = sum(exps.values())
        return {c: e / total for c, e in exps.items()}

    def predict(self, X: list[list[float | None]]) -> list[str]:
        result = []
        for x in X:
            scores = self.log_scores(x)
            result.append(max(scores, key=scores.get))
        return result

    def to_dict(self) -> dict:
        return {
            "var_smoothing": self.var_smoothing,
            "classes": self.classes_,
            "priors": self.priors_,
            "means": self.means_,
            "variances": self.vars_,
        }

    @classmethod
    def from_dict(cls, d: dict) -> "GaussianNB":
        model = cls(var_smoothing=d["var_smoothing"])
        model.classes_ = d["classes"]
        model.priors_ = d["priors"]
        model.means_ = d["means"]
        model.vars_ = d["variances"]
        return model

A few things worth pointing out:

- **`var_smoothing`** adds a tiny amount to every variance. If a feature happened to have exactly the same value for every training example of a class, its variance would be 0, and the formula would divide by zero. scikit-learn's `GaussianNB` uses the same idea and the same default.
- **Missing values are handled naturally.** If a measurement is `None`, that species' score simply doesn't get a term for it. The model makes its decision from the measurements it does have. Not every algorithm can do this. k-NN, for example, can't compute a distance with a missing coordinate.
- **`predict_proba`** turns the log scores into probabilities with softmax and the max-subtraction trick from the Module 2 spam filter.
- **The class names are strings**, so `to_dict` produces JSON with string keys, and nothing gets mangled on the round trip (the key problem from the lesson only affects non-string keys).

Train it and look at what it learned:

In [5]:
model = GaussianNB().fit(X_train, y_train)

print(f"{'':10s}{'prior':>7s}" + "".join(f"{f.rsplit('_', 1)[0]:>16s}" for f in FEATURES))
for c in model.classes_:
    means = "".join(f"{m:16.1f}" for m in model.means_[c])
    print(f"{c:10s}{model.priors_[c]:7.2f}{means}")

            prior     bill_length      bill_depth  flipper_length       body_mass
Adelie       0.44            38.8            18.4           189.9          3701.5
Chinstrap    0.20            49.0            18.5           195.8          3768.1
Gentoo       0.36            47.7            15.0           217.3          5088.3


That table *is* the model (plus the variances). Every prediction it makes comes from these numbers. Compare it to the per-species averages from the lesson: they're nearly identical, since this is the same calculation on 75% of the data.

## Step 5: Evaluate

First, overall accuracy, alongside two other models for comparison: the majority-class baseline, and k-NN with scaled features from earlier modules.

In [6]:
def accuracy(y_true, y_pred) -> float:
    return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)


majority = Counter(y_train).most_common(1)[0][0]
baseline_pred = [majority] * len(y_test)


def knn_predict(X_tr, y_tr, X_te, k=5):
    cols = list(zip(*X_tr))
    means = [sum(c) / len(c) for c in cols]
    stds = [(sum((v - m) ** 2 for v in c) / len(c)) ** 0.5 for c, m in zip(cols, means)]
    scale = lambda x: [(v - m) / s for v, m, s in zip(x, means, stds)]
    train_scaled = [scale(x) for x in X_tr]
    preds = []
    for x in X_te:
        xs = scale(x)
        nearest = sorted(zip(train_scaled, y_tr), key=lambda p: sum((a - b) ** 2 for a, b in zip(p[0], xs)))[:k]
        preds.append(Counter(label for _, label in nearest).most_common(1)[0][0])
    return preds


nb_pred = model.predict(X_test)
knn_pred = knn_predict(X_train, y_train, X_test)

print(f"majority baseline: {accuracy(y_test, baseline_pred):.1%}")
print(f"k-NN (scaled, k=5): {accuracy(y_test, knn_pred):.1%}")
print(f"Gaussian NB:        {accuracy(y_test, nb_pred):.1%}")

majority baseline: 44.2%
k-NN (scaled, k=5): 100.0%
Gaussian NB:        97.7%


Both real models are far above the baseline. On this split, k-NN gets every test penguin right and Naive Bayes misses two.

Part of the reason is the "naive" assumption. Within a species, the measurements are correlated: bigger penguins tend to have longer flippers *and* heavier bodies *and* longer bills. Naive Bayes treats them as independent, so it effectively counts the same "this is a big penguin" evidence several times over. k-NN makes no such assumption.

Still, keep the numbers in proportion. With 86 test penguins, each mistake is worth 1.2 percentage points, so the difference between the two models is just two penguins. A different random split could easily reverse it. Naive Bayes has other strengths that matter in practice, as the next steps show: it handles missing measurements, it gives meaningful probabilities, and the whole model is a small table of numbers you can read.

Now a confusion matrix, to see *which* species get confused when the model is wrong:

In [7]:
def confusion_matrix(y_true, y_pred, labels):
    counts = Counter(zip(y_true, y_pred))
    width = max(len(l) for l in labels) + 2
    print(" " * width + "".join(f"{l:>{width}}" for l in labels) + "   <- predicted")
    for t in labels:
        print(f"{t:>{width}}" + "".join(f"{counts[(t, p)]:>{width}}" for p in labels))


confusion_matrix(y_test, nb_pred, model.classes_)

print()
for c in model.classes_:
    tp = sum(t == c and p == c for t, p in zip(y_test, nb_pred))
    predicted = sum(p == c for p in nb_pred)
    actual = sum(t == c for t in y_test)
    print(f"{c:10s} precision={tp / predicted:.2f}  recall={tp / actual:.2f}")

                Adelie  Chinstrap     Gentoo   <- predicted
     Adelie         38          0          0
  Chinstrap          2         15          0
     Gentoo          0          0         31

Adelie     precision=0.95  recall=1.00
Chinstrap  precision=1.00  recall=0.88
Gentoo     precision=1.00  recall=1.00


`Counter(zip(y_true, y_pred))` counts every (true, predicted) pair in one line, and since it's a `Counter`, pairs that never happened return 0.

The confusion matrix shows where the errors are. If there are any, they're almost certainly between Adelie and Chinstrap, which (as we saw in the lesson) are similar in everything except bill length. Gentoos, with their much larger bodies and flippers, are hard to mistake for anything else.

Let's look more closely at the two mistakes, and at the penguins the model was least sure about:

In [8]:
results = []
for row, x in zip(test, X_test):
    probs = model.predict_proba(x)
    pred = max(probs, key=probs.get)
    results.append((probs[pred], row["species"], pred, x))

print("mistakes:")
for conf, true, pred, x in results:
    if true != pred:
        print(f"  {conf:6.1%} sure  true={true:9s} pred={pred:9s} {x}")

print("least confident:")
for conf, true, pred, x in sorted(results)[:4]:
    mark = "correct" if true == pred else "WRONG"
    print(f"  {conf:6.1%} sure  true={true:9s} pred={pred:9s} {mark}  {x}")

mistakes:
   94.5% sure  true=Chinstrap pred=Adelie    [42.5, 17.3, 187.0, 3350.0]
   95.9% sure  true=Chinstrap pred=Adelie    [43.2, 16.6, 187.0, 2900.0]
least confident:
   55.2% sure  true=Chinstrap pred=Chinstrap correct  [45.2, 16.6, 191.0, 3250.0]
   74.7% sure  true=Adelie    pred=Adelie    correct  [43.2, 18.5, 192.0, 4100.0]
   81.1% sure  true=Adelie    pred=Adelie    correct  [41.1, 18.1, 205.0, 4300.0]
   83.0% sure  true=Adelie    pred=Adelie    correct  [42.2, 19.5, 197.0, 4275.0]


This is worth reading carefully. The model was quite sure about both of its mistakes. They're Chinstraps with bills on the short side for their species, which puts them squarely in Adelie territory. Meanwhile, its least confident predictions all turned out to be correct.

Two lessons come out of this. First, **confidence is not the same as correctness**. A probability of 90% means "this looks a lot like the Adelies I was trained on", not "this is right 90% of the time". Naive Bayes in particular is known for overconfident probabilities, again because of the independence assumption. Second, **look at your errors one by one**, not just at the overall score. Here, it tells us the errors aren't random noise: they're real penguins whose measurements overlap with another species. No amount of tuning will fix that with these four features, but a new feature might, which is what the first "Your turn" idea explores.

## Step 6: Save the model

Now we save the model as a JSON file. Alongside the learned numbers, we store **metadata**: everything someone would need to use the model correctly or to reproduce it. Which features it expects, and in what order. The data's checksum. The split seed. The test score.

In [9]:
def save_json(obj, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2) + "\n", encoding="utf-8")
    tmp.replace(path)
    return path


model_file = {
    "format_version": 1,
    "model_type": "GaussianNB",
    "features": FEATURES,
    "training_data": {
        "file": raw_path.name,
        "sha256": data_hash,
        "rows_used": len(usable),
        "split_seed": SEED,
        "test_ratio": 0.25,
    },
    "test_accuracy": round(accuracy(y_test, nb_pred), 4),
    "model": model.to_dict(),
}

model_path = save_json(model_file, WORK / "models" / "penguin_nb.json")
print(model_path, f"({model_path.stat().st_size:,} bytes)")
print(model_path.read_text(encoding="utf-8")[:700], "...")

work/models/penguin_nb.json (1,482 bytes)
{
  "format_version": 1,
  "model_type": "GaussianNB",
  "features": [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
  ],
  "training_data": {
    "file": "penguins.csv",
    "sha256": "f204db2c753b0937caac3cb35258562c14f073e4bbc76be24b4c51ce22767a93",
    "rows_used": 342,
    "split_seed": 7,
    "test_ratio": 0.25
  },
  "test_accuracy": 0.9767,
  "model": {
    "var_smoothing": 1e-09,
    "classes": [
      "Adelie",
      "Chinstrap",
      "Gentoo"
    ],
    "priors": {
      "Adelie": 0.44140625,
      "Chinstrap": 0.19921875,
      "Gentoo": 0.359375
    },
    "means": {
      "Adelie": [
        38.78849557522124,
        18.39646017699115,
 ...


The whole model is a few kilobytes of readable text. You can open it, see every number the model uses, and put it under version control so `git diff` shows exactly what changed when you retrain. You could even load it from another programming language.

`save_json` writes to a temporary file and then renames it, the atomic-write trick from exercise 12, so a crash halfway through saving can never leave a half-written model file behind.

`format_version` is there for the future. If you later change what the file contains, the loading code can check the version and handle old files properly instead of failing in a confusing way.

## Step 7: Load it back

Now, pretend we're a different program, say a web service, that only has the model file. It knows nothing about the training data or the `model` variable above:

In [10]:
def load_model(path: Path) -> tuple[GaussianNB, dict]:
    saved = json.loads(path.read_text(encoding="utf-8"))
    if saved.get("format_version") != 1 or saved.get("model_type") != "GaussianNB":
        raise ValueError(f"{path} is not a model file this code can read")
    return GaussianNB.from_dict(saved["model"]), saved


loaded, info = load_model(model_path)

print("expects features:", info["features"])
print("trained on data with sha256:", info["training_data"]["sha256"][:16], "...")

same = loaded.predict(X_test) == model.predict(X_test)
print("identical predictions to the original:", same)

expects features: ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
trained on data with sha256: f204db2c753b0937 ...
identical predictions to the original: True


Identical predictions, from a model rebuilt entirely from a text file. The loader also checks the file is something it understands before using it. Loading an unexpected file, such as an older format, a different model type, or the wrong file altogether, gives a clear error instead of wrong predictions.

And because we saved the data's checksum, we can check that the data on disk is still the data the model was trained on:

In [11]:
if sha256_of(raw_path) == info["training_data"]["sha256"]:
    print("the data file matches the one the model was trained on")
else:
    print("WARNING: the data file has changed since this model was trained")

the data file matches the one the model was trained on


## Step 8: Predict for new penguins

Finally, the real job: new observations arrive as a CSV, perhaps from a field researcher's tablet. Some measurements are missing. We read them, predict, and write a new CSV with the predicted species and the probability of each.

In [12]:
new_observations = """tag,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
N01,39.5,18.8,190,3800
N02,49.9,15.1,222,5300
N03,50.2,19.0,198,3700
N04,46.0,NA,210,
N05,44.0,18.0,195,3900
"""
new_path = WORK / "incoming" / "new_penguins.csv"
new_path.parent.mkdir(parents=True, exist_ok=True)
new_path.write_text(new_observations, encoding="utf-8")


def predict_file(model: GaussianNB, features: list[str], in_path: Path, out_path: Path) -> list[dict]:
    with open(in_path, encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        absent = set(features) - set(reader.fieldnames)
        if absent:
            raise ValueError(f"{in_path} is missing columns: {sorted(absent)}")
        out_rows = []
        for raw in reader:
            x = [to_number(raw[col]) for col in features]
            probs = model.predict_proba(x)
            row = dict(raw)
            row["predicted_species"] = max(probs, key=probs.get)
            row["n_missing"] = sum(v is None for v in x)
            for c in model.classes_:
                row[f"p_{c}"] = round(probs[c], 4)
            out_rows.append(row)

    with open(out_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(out_rows[0]))
        writer.writeheader()
        writer.writerows(out_rows)
    return out_rows


out_path = WORK / "predictions" / "new_penguins_predicted.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
predict_file(loaded, info["features"], new_path, out_path)

print(out_path.read_text(encoding="utf-8"))

tag,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,predicted_species,n_missing,p_Adelie,p_Chinstrap,p_Gentoo
N01,39.5,18.8,190,3800,Adelie,0,0.9927,0.0073,0.0
N02,49.9,15.1,222,5300,Gentoo,0,0.0,0.0,1.0
N03,50.2,19.0,198,3700,Chinstrap,0,0.0002,0.9998,0.0
N04,46.0,NA,210,,Gentoo,2,0.0008,0.0951,0.9041
N05,44.0,18.0,195,3900,Chinstrap,0,0.4597,0.5403,0.0



Reading the results:

- **N01** has short bill, deep bill, short flippers: a confident Adelie.
- **N02** is large, with long flippers and a shallow bill: a confident Gentoo.
- **N03** has a long bill on a small body: a Chinstrap.
- **N04** has two missing measurements, but the model still made a prediction from the other two. The `n_missing` column records that, so whoever uses these predictions can treat it with appropriate care.
- **N05** sits between the typical Adelie and Chinstrap measurements. Look at how its probability is split, compared to the others.

The prediction step reads the feature list from the saved model file (`info["features"]`), not from the `FEATURES` variable in this notebook. That's deliberate: the model file is the single source of truth about what the model expects, so the two can never drift apart.

## Your turn

1. **Use the island.** The island is a strong clue (in this data, Gentoos were only seen on Biscoe and Chinstraps only on Dream). Naive Bayes can mix feature types: multiply in a *categorical* likelihood for the island, estimated by counting as in the spam filter, with Laplace smoothing. Does accuracy improve? Is it a good idea to rely on location if the model will be used on other islands?

2. **Use fewer features.** Train on every pair of features (with `itertools.combinations` from Module 4). Which pair gives the best accuracy? What does that tell you about which measurements actually separate the species?

3. **A model registry on disk.** Save each trained model as `models/penguin_nb_v{n}.json`, and write a function that lists the saved versions with their test accuracy and data checksum, reading only the JSON files.

4. **Log every prediction.** Append each prediction from `predict_file` to a JSON Lines log (`predictions.jsonl`) with a timestamp and the model's checksum. Logs like this are how deployed models are monitored. For example, if the share of low-confidence predictions starts rising, the incoming data may have changed.

5. **Validate the input.** Use `validate_records` from exercise 10, with sensible min/max values per feature, to reject impossible observations (like a negative flipper length) before predicting.

6. **Compare with scikit-learn.** If you have scikit-learn available (Colab does), train `sklearn.naive_bayes.GaussianNB` on the same split and check that its predictions match yours.

## What you used

- **`pathlib`** for every path, with `mkdir(parents=True, exist_ok=True)`
- **Download-once caching** for the dataset
- **`csv.DictReader` and `csv.DictWriter`** with `newline=""` and `"utf-8-sig"`, header validation, and explicit missing-value handling
- **`hashlib.sha256`** in binary chunks to fingerprint the training data
- **A stratified, seeded split** for reproducibility
- **JSON** for the model file, with metadata, a format version, and an atomic write
- **`to_dict` / `from_dict`** to convert a model to and from plain data
- **Log probabilities and softmax** from Module 2, and **`Counter`** for the confusion matrix

And the ML life cycle: load, validate, split, train, evaluate against baselines, inspect uncertain cases, save with metadata, load elsewhere, and predict on new data with missing values, keeping a record of what was missing.